In [1]:
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt
import pandas as pd
from skimage.measure import regionprops, label
from natsort import natsorted
from cellpose import plot
import trackpy as tp
import sys
sys.path.append('../defect_functions') 
from defect_pairs import * 
from average_flows import * 

# Set up matplotlib to use the Qt backend
%matplotlib qt

def calculate_neighbors(mask_image, area_threshold=60, boundary_margin=5):
    """
    Calculate neighbors and centroids for each region in the mask image.
    
    Parameters:
    - mask_image: The binary mask image where regions are labeled.
    - area_threshold: Minimum area for a region to be considered (default is 60).
    - boundary_margin: Margin from the image boundary to exclude contours (default is 5).
    
    Returns:
    - neighbors: A dictionary containing the centroids and neighboring cells for each region.
    """
    height, width = mask_image.shape
    neighbors = {}
    
    # Get region properties
    regions = regionprops(mask_image, intensity_image=mask_image)
    
    for region in regions:
        area = region.area
        color = int(region.mean_intensity)
        
        if area > area_threshold:
            mask = np.uint8(mask_image == color)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                centroid = region.centroid  # Get the centroid of the region
                area = region.area
                perimeter = region.perimeter
                if color not in neighbors:
                    neighbors[color] = {'Centroid': centroid, 
                                        'Area': area, 
                                        'Perimeter': perimeter, 
                                        'Neighboring Cells': set()}

                for contour in contours:
                    # Check if the contour is within the boundary margin
                    if all(boundary_margin <= point[0][0] < width - boundary_margin and 
                           boundary_margin <= point[0][1] < height - boundary_margin for point in contour):                    
                
                    # for contour in contours:
                        for point in contour:
                            x, y = point[0]
                            # Check neighboring pixels
                            for i in range(max(0, y - 1), min(height, y + 2)):
                                for j in range(max(0, x - 1), min(width, x + 2)):
                                    if mask_image[i, j] != color and mask_image[i, j] != 0:
                                        neighbors[color]['Neighboring Cells'].add(mask_image[i, j])
    return neighbors


def create_dataframe(neighbors):
    """
    Create a DataFrame from the neighbors dictionary, excluding empty neighbors.
    
    Parameters:
    - neighbors: A dictionary containing neighboring cells and centroids.
    
    Returns:
    - A pandas DataFrame with cells, their centroids, number of neighbors, and the list of neighbors.
    """
    data = []
    for k, v in neighbors.items():
        if v['Neighboring Cells']:  # Only include if there are neighboring cells
            data.append({'Cell': k, 
                         'x': v['Centroid'][1], 
                         'y': v['Centroid'][0], 
                         'Area': v['Area'], 
                         'Perimeter': v['Perimeter'], 
                         'Neighbors Num': len(v['Neighboring Cells']), 
                         'Neighbors': list(v['Neighboring Cells'])})
    return pd.DataFrame(data)


def create_area_mask(mask_image):
    """
    Create a new mask labeled by region area.
    
    Parameters:
    - mask_image: The binary mask image where regions are labeled.
    
    Returns:
    - area_mask: A new mask where each region is labeled by its area.
    """
    # Label the regions in the mask image
    labeled_mask = label(mask_image)
    area_mask = np.zeros_like(labeled_mask, dtype=np.uint16)  # Create an empty mask for areas

    # Get region properties
    regions = regionprops(labeled_mask)

    # Assign area values to the new mask
    for region in regions:
        # area = int(255*region.eccentricity)
        area = region.area
        # Fill the area_mask with the area value for the corresponding region
        area_mask[labeled_mask == region.label] = area

    return area_mask

In [12]:
# Main execution
im_num = 100
image_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\*.tif"
mask_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\Mask2\*.png"

img_list = natsorted(glob.glob(image_path), key=lambda y: y.lower())
masks_list = natsorted(glob.glob(mask_path), key=lambda y: y.lower())

# Load images
x, y, w, h = [0, 0, 800, 500]
raw_image = cv2.imread(img_list[im_num], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
mask_image = cv2.imread(masks_list[im_num], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]    

PLOT = False
if PLOT:
    # Display overlay
    plt.figure()
    overlay = cv2.addWeighted(cv2.cvtColor(raw_image, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.5, 0)
    plt.imshow(overlay)

    # Display overlay with area values
    area = [region.area for region in regionprops(mask_image)]
    perimeter = [region.perimeter for region in regionprops(mask_image)]
    area_value = [int(region.mean_intensity) for region in regionprops(mask_image, intensity_image=mask_image)]
    xy = np.array([(region.centroid[1], region.centroid[0]) for region in regionprops(mask_image)])
    plt.plot(xy[:,0], xy[:,1], 'w.', alpha=.3)
    for i in range(len(xy)):
        col = 'k' if area[i]<60 else 'w'
        plt.text(xy[i,0], xy[i,1], f"{int(area_value[i])}", color=col, fontsize=14)

# Calculate neighbors
neighbors = calculate_neighbors(mask_image)

# Create DataFrame
df_neighbors = create_dataframe(neighbors)

# # Optionally save DataFrame to CSV
# # df_neighbors.to_csv('neighbors_data.csv', index=False)

# # Display the DataFrame
# # print(df_neighbors)
# for index, row in df_neighbors.iterrows():
#     plt.text(row['Centroid'][1], row['Centroid'][0], f"{int(row['Num'])}", color='w', fontsize=12)

In [13]:
# Main execution
# Display overlay
plt.figure()
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe.apply(raw_image)
# plt.imshow(255-img_clahe, cmap='gray')
overlay = cv2.addWeighted(cv2.cvtColor(raw_image, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.3, 0)
plt.imshow(overlay)
# plt.imshow(create_area_mask(mask_image), "coolwarm")

for index, row in df_neighbors.iterrows():
    plt.text(row['x'], row['y'], f"{int(row['Neighbors Num'])}", color='w', fontsize=12)

In [14]:
img1 = cv2.imread(img_list[im_num-1], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
img2 = cv2.imread(img_list[im_num], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
flow = cv2.calcOpticalFlowFarneback(img1,img2, None, 0.5, 3, 
        winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0) 

step = 30
x = np.arange(0, flow.shape[1], step, dtype=np.int16)
y = np.arange(0, flow.shape[0], step, dtype=np.int16)
plt.quiver(x,y, 
    flow[::step, ::step, 0], -flow[::step, ::step, 1], 
    color="k", scale=100, label="flow", alpha=.5)

In [4]:
s = 30
ori,_,_ = analyze_defects(raw_image, sigma=25)
y, x = np.mgrid[0:img1.shape[0], 0:img1.shape[1]]
quiver = plt.quiver(x[::s,::s], y[::s,::s],
    np.cos(ori)[::s,::s], np.sin(ori)[::s,::s], 
    # np.arctan2(np.sin(ori), np.cos(ori))[::s,::s],
    headaxislength=0, headwidth=0, headlength=0, width=.005, 
    scale=30, pivot='mid', alpha=.6)

In [5]:
df_neighbors.x.max(), df_neighbors.y.max()

(np.float64(1596.1768707482993), np.float64(1091.9906542056074))

In [10]:
import seaborn as sns
data = df_neighbors[df_neighbors["Area"]>60].copy()

plt.figure()
data["Shape"] = data["Perimeter"]**2/(4*np.pi*data["Area"])
sns.pairplot(data[["Area", "Perimeter", "Neighbors Num", "Shape"]])

In [11]:
data[["Area", "Perimeter", "Neighbors Num", "Shape"]].corr().style.background_gradient(cmap='coolwarm')

,Area,Perimeter,Neighbors Num,Shape
Area,1.000000,0.946376,0.514222,0.188042
Perimeter,0.946376,1.000000,0.501700,0.472692
Neighbors Num,0.514222,0.501700,1.000000,0.119279
Shape,0.188042,0.472692,0.119279,1.000000


In [107]:
image_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\*.tif"
mask_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\Mask2\*.png"

img_list = natsorted(glob.glob(image_path), key=lambda y: y.lower())
masks_list = natsorted(glob.glob(mask_path), key=lambda y: y.lower())
x, y, w, h = [0, 0, 500, 500]

df_list = []
for frame, mask_path in enumerate(masks_list[-20:]):
    # Calculate neighbors
    neighbors = calculate_neighbors(cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w])
    

    # Create DataFrame
    df_neighbors = create_dataframe(neighbors)
    df_neighbors["frame"] = frame
    df_list.append(df_neighbors)
    # break

features = pd.concat(df_list, ignore_index=True)

search_range = 7 #15
t = tp.link_df(features, search_range, memory=1)

Frame 19: 1108 trajectories present.


In [126]:
t1 = t.groupby('particle').filter(lambda x: len(x)>7).copy()

In [127]:
t1.shape

(18664, 9)

In [129]:
t1["Cell_tracked"] = t1["particle"]

# Create a mapping from Cell values to Cell_tracked values
cell_to_tracked = dict(zip(t1['Cell'], t1['Cell_tracked']))

# # Replace the values in the Neighbors column
# t1['Neighbors_tracked'] = t1['Neighbors'].transform(lambda neighbors: [cell_to_tracked.get(int(neighbor), int(neighbor)) for neighbor in neighbors])

# Function to remap Neighbors for each frame
def remap_neighbors(group):
    # Create a mapping from Cell to Cell_tracked for the current group
    cell_to_tracked = dict(zip(group['Cell'], group['Cell_tracked']))
    # Replace the values in the Neighbors column
    group['Neighbors'] = group['Neighbors'].apply(lambda neighbors: [cell_to_tracked.get(neighbor, neighbor) for neighbor in neighbors])
    return group

# Apply the remapping function to each group
t1 = t1.groupby('frame').apply(remap_neighbors)

C:\Users\victo\AppData\Local\Temp\ipykernel_27580\3901880585.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t1 = t1.groupby('frame').apply(remap_neighbors)


In [125]:
t1[t1["frame"]==10]

Cell           x          y   Area  Perimeter  Neighbors Num  \
frame                                                                       
10    10909   131  386.000000  10.935897   78.0  33.627417              5   
      10939   217   43.771739  22.809783  184.0  54.284271              5   
      10937   208  182.600000  22.045714  175.0  49.556349              5   
      10968   312  228.576087  35.614130  184.0  54.041631              7   
      10967   306  417.529825  36.926316  285.0  67.870058              7   
...           ...         ...        ...    ...        ...            ...   
      10948   247  420.777027  25.040541  148.0  47.213203              5   
      10949   258  284.263844  30.413681  307.0  68.426407              8   
      10916   146  272.409639  15.626506  249.0  62.870058              8   
      10917   147  299.534483  12.810345  116.0  41.142136              5   
      10923   161  350.237410  17.503597  139.0  47.213203              7   

                                           Neighbors  frame  particle  \
frame                                                                   
10    10909                   [32, 33, 197, 19, 187]     10         9   
      10939               [1, 1376, 1305, 1543, 145]     10      1208   
      10937             [1049, 15, 1080, 1068, 1378]     10        29   
      10968        [90, 153, 59, 1061, 62, 13, 1152]     10      1308   
      10967          [65, 81, 15, 110, 71, 55, 1546]     10        50   
...                                              ...    ...       ...   
      10948                    [65, 36, 15, 50, 188]     10        55   
      10949   [282, 57, 102, 83, 41, 1422, 382, 159]     10      1056   
      10916  [1056, 1082, 21, 22, 23, 1422, 58, 159]     10        41   
      10917                  [57, 1372, 24, 25, 159]     10      1044   
      10923         [52, 294, 236, 1381, 28, 29, 30]     10        27   

             Cell_tracked  
frame                      
10    10909             9  
      10939          1208  
      10937            29  
      10968          1308  
      10967            50  
...                   ...  
      10948            55  
      10949          1056  
      10916            41  
      10917          1044  
      10923            27  

[1020 rows x 10 columns]

In [130]:
df = t1[t1["Cell_tracked"]==56]
for index, row in df.iterrows():
    print(">> " + str(index))
    for n in row['Neighbors']:
        # print(n)
        print(t1[t1["Cell_tracked"]==n][["x","y"]])

>> (0, 119)
                      x           y
frame                              
0     650    181.434010  295.708122
1     1736   180.035000  293.757500
2     2813   181.500000  291.834286
3     3895   183.149560  288.618768
4     4971   179.071749  286.161435
5     6059   178.215447  281.825203
6     7159   177.521008  279.579832
7     8258   173.024845  276.565217
8     9350   173.908046  276.103448
9     10447  169.702381  276.910714
10    11525  166.734463  279.000000
11    12610  162.355072  279.927536
12    13695  159.395522  278.738806
13    14770  163.785408  279.690987
14    15861  161.748768  279.231527
15    16946  157.366906  281.194245
16    18011  156.344262  281.368852
17    19096  153.267606  281.485915
18    20189  152.251908  281.351145
19    21279  148.136691  280.906475
                      x           y
frame                              
0     516    416.524590  227.819672
1     1598   418.854460  227.056338
2     2679   421.790909  226.418182
3     3763   424

In [123]:
mask_image = cv2.imread(masks_list[-20], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]

plt.figure()

plt.imshow(create_area_mask(mask_image), "coolwarm")

df = t1[t1["Cell_tracked"]==56]
for index, row in df.iterrows():
    print(">> " + str(index))
    for n in row['Neighbors']:
        # print(n)
        print(t1[t1["Cell_tracked"]==n][["x","y"]])
        plt.text(row['x'], row['y'], f"{int(row['Neighbors Num'])}", color='w', fontsize=12)
        plt.plot(t1[t1["Cell_tracked"]==n]['x'], t1[t1["Cell_tracked"]==n]['y'], "+k")

# for index, row in t1[t1["Cell_tracked"]==56].iterrows():
#     plt.text(row['x'], row['y'], f"{int(row['Neighbors Num'])}", color='w', fontsize=12)
#     plt.plot(row['x'], row['y'], "ok")

>> (0, 119)
                      x           y
frame                              
0     650    181.434010  295.708122
1     1736   180.035000  293.757500
2     2813   181.500000  291.834286
3     3895   183.149560  288.618768
4     4971   179.071749  286.161435
5     6059   178.215447  281.825203
6     7159   177.521008  279.579832
7     8258   173.024845  276.565217
8     9350   173.908046  276.103448
9     10447  169.702381  276.910714
10    11525  166.734463  279.000000
11    12610  162.355072  279.927536
12    13695  159.395522  278.738806
13    14770  163.785408  279.690987
14    15861  161.748768  279.231527
15    16946  157.366906  281.194245
16    18011  156.344262  281.368852
17    19096  153.267606  281.485915
18    20189  152.251908  281.351145
19    21279  148.136691  280.906475
                      x           y
frame                              
0     516    416.524590  227.819672
1     1598   418.854460  227.056338
2     2679   421.790909  226.418182
3     3763   424

In [90]:
# search_range = 7 #15
# t = tp.link_df(features, search_range, memory=1)
# plt.imshow(cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w])
# tp.plot_traj(t)

In [101]:
# x, y, w, h = [0, 0, 800, 800]
# mask_image = cv2.imread(masks_list[-10], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# img1 = cv2.imread(img_list[-10], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# img2 = cv2.imread(img_list[-9], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# flow = cv2.calcOpticalFlowFarneback(img1,img2, None, 0.5, 3, 
#         winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0) 

# plt.figure()
# # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
# # img_clahe = clahe.apply(img1)
# # plt.imshow(255-img_clahe, cmap='gray')
# overlay = cv2.addWeighted(cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.4, 0)
# plt.imshow(overlay)

# step = 15
# xx = np.arange(0, flow.shape[1], step, dtype=np.int16)
# yy = np.arange(0, flow.shape[0], step, dtype=np.int16)
# plt.quiver(xx,yy, 
#     flow[::step, ::step, 0], -flow[::step, ::step, 1], 
#     color="w", scale=100, label="flow", alpha=.6)

# tp.plot_traj(t)

<Axes: xlabel='x [px]', ylabel='y [px]'>

In [92]:
t["dx"] = t.groupby("particle")["x"].diff()
t["dy"] = t.groupby("particle")["y"].diff()
t["displacement"] = np.sqrt(t["dx"]**2 + t["dy"]**2)
t.dropna(inplace=True)

In [96]:
import seaborn as sns
# data = t[t["Area"]>60].copy()
data = t.copy()
data["Shape"] = data["Perimeter"]**2/(4*np.pi*data["Area"])
sns.pairplot(data[["Area", "Perimeter", "Shape", "Neighbors Num", "displacement"]])

In [102]:
data[["Area", "Perimeter", "Shape", "Neighbors Num", "displacement"]].corr().style.background_gradient(cmap='coolwarm')

,Area,Perimeter,Shape,Neighbors Num,displacement
Area,1.000000,0.942801,0.207784,0.474773,-0.051676
Perimeter,0.942801,1.000000,0.500443,0.462450,-0.039247
Shape,0.207784,0.500443,1.000000,0.113627,0.027518
Neighbors Num,0.474773,0.462450,0.113627,1.000000,-0.008565
displacement,-0.051676,-0.039247,0.027518,-0.008565,1.000000
